# E-Commerce Sales & Delivery Performance Analysis
### Olist Brazilian E-Commerce Dataset

**Business question:** What's driving late deliveries and low review scores, and which product categories/regions should the business prioritize fixing first?

**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) | ~100k orders, 2016–2018.

**Approach:** SQL (via DuckDB, run directly over the raw CSVs) for extraction/joins, pandas for cleaning and analysis, matplotlib/seaborn/plotly for visualization.

**Grain warning:** `order_items` is **item-grain** (one row per item, not per order). `orders`/`delivered` is **order-grain**. Never join `order_items` directly into order-level review/delay analysis without first aggregating to one row per order | doing so silently duplicates review scores across every item in a multi-item order.

---


## 1. Setup

In [ ]:
# If duckdb isn't preinstalled in this Kaggle environment, uncomment:
# !pip install duckdb -q

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


In [ ]:
# Path to the dataset once attached to this Kaggle Notebook.
# Kaggle's actual path can vary | if this fails, run:
#   import os; print(os.listdir("/kaggle/input"))
# and walk down until you find the CSVs, then update DATA_DIR.
DATA_DIR = "/kaggle/input/brazilian-ecommerce"

import os
print(os.listdir(DATA_DIR))


## 2. Load Data

DuckDB can query CSV files directly with SQL — no import/load step needed. `con.execute(...).df()` runs a SQL query and returns a pandas DataFrame.

In [ ]:
con = duckdb.connect()

orders = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}/olist_orders_dataset.csv')
""").df()

order_items = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}/olist_order_items_dataset.csv')
""").df()

products = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}/olist_products_dataset.csv')
""").df()

customers = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}/olist_customers_dataset.csv')
""").df()

sellers = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}/olist_sellers_dataset.csv')
""").df()

reviews = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}/olist_order_reviews_dataset.csv')
""").df()

payments = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}/olist_order_payments_dataset.csv')
""").df()

category_translation = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}/product_category_name_translation.csv')
""").df()

print("orders:", orders.shape, "(grain: 1 row per order)")
print("order_items:", order_items.shape, "(grain: 1 row per item | DO NOT join into order-level analysis raw)")
print("products:", products.shape)
print("customers:", customers.shape)
print("sellers:", sellers.shape)
print("reviews:", reviews.shape)
print("payments:", payments.shape)


## 3. Explore Before Analyzing

Row counts, date ranges, nulls, and how tables relate — do this before writing any "real" analysis query.

In [ ]:
orders.info()
orders.head()


In [ ]:
orders["order_status"].value_counts()


In [ ]:
print(orders["order_purchase_timestamp"].min(), "to", orders["order_purchase_timestamp"].max())


In [ ]:
orders[["order_delivered_customer_date", "order_estimated_delivery_date"]].isna().sum()


## 4. Clean Data

- Keep only `delivered` orders for delivery-time analysis.
- Convert date columns to datetime.
- Drop duplicate reviews.
- Aggregate `order_items` to **order grain** before it touches any order-level analysis (freight cost, item count) — this directly fixes the grain risk flagged above.

In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

delivered = orders[orders["order_status"] == "delivered"].copy()
print(f"{len(delivered)} of {len(orders)} orders are 'delivered' and usable for delivery-time analysis")


In [ ]:
reviews_before = len(reviews)
reviews = reviews.drop_duplicates(subset="review_id")
print(f"Dropped {reviews_before - len(reviews)} duplicate review rows")


In [ ]:
products = products.merge(category_translation, on="product_category_name", how="left")
products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")


In [ ]:
# Aggregate order_items to ORDER grain (not item grain) before using it downstream.
# One row per order: total freight, total item price, item count, and a single
# "primary category" (the category of the highest-value item in the order |
# a reasonable simplification for multi-category orders, which are a minority).
order_items_agg = (
    order_items.merge(products[["product_id", "product_category_name_english"]], on="product_id", how="left")
    .sort_values("price", ascending=False)
    .groupby("order_id")
    .agg(
        n_items=("order_item_id", "count"),
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        primary_category=("product_category_name_english", "first"),  # highest-price item, due to sort above
    )
    .reset_index()
)
print(f"order_items_agg: {order_items_agg.shape} | now one row per order, safe to join into order-level analysis")


## 5. Q1 — Which categories drive revenue vs. order volume? (Pareto analysis)

In [ ]:
revenue_by_category = con.execute("""
    SELECT
        p.product_category_name_english AS category,
        COUNT(DISTINCT oi.order_id)     AS n_orders,
        SUM(oi.price)                   AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY 1
    ORDER BY total_revenue DESC
""").df()

revenue_by_category["revenue_share"] = revenue_by_category["total_revenue"] / revenue_by_category["total_revenue"].sum()
revenue_by_category["cum_revenue_share"] = revenue_by_category["revenue_share"].cumsum()
revenue_by_category.head(15)


In [ ]:
top15 = revenue_by_category.head(15)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top15, y="category", x="total_revenue", ax=ax, color="#4C72B0")
ax.set_title("Top 15 Categories by Revenue")
ax.set_xlabel("Total Revenue (BRL)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

n_categories_for_80pct = (revenue_by_category["cum_revenue_share"] <= 0.8).sum()
print(f"{n_categories_for_80pct} of {len(revenue_by_category)} categories account for ~80% of revenue")


## 6. Q2 — Delivery Delay vs. Review Score

Delay = actual delivery date − estimated delivery date. Positive = late.

**Fixes applied from audit:**
- Using **Spearman** correlation, not Pearson — review score is ordinal (1–5), not continuous, so Spearman is the statistically appropriate choice.
- Reporting the % of orders dropped by `dropna` so the correlation's scope is transparent.
- Outlier sanity check on delay before trusting the chart's scale.
- **Segmented by category** — this is the critical fix. Without it, we can't tell whether delay itself drives review score, or whether certain categories are simply both slower *and* lower-rated (a Simpson's-paradox risk flagged in the audit).

In [ ]:
delivered_reviewed = delivered.merge(
    reviews[["order_id", "review_score"]], on="order_id", how="left"
)
delivered_reviewed["delivery_delay_days"] = (
    delivered_reviewed["order_delivered_customer_date"] - delivered_reviewed["order_estimated_delivery_date"]
).dt.days

n_before = len(delivered_reviewed)
delay_review = delivered_reviewed.dropna(subset=["delivery_delay_days", "review_score"])
n_after = len(delay_review)
print(f"Dropped {n_before - n_after} rows ({(n_before - n_after) / n_before:.1%}) missing delay or review score")
print(f"Analysis covers {n_after} orders")

delay_review[["delivery_delay_days", "review_score"]].describe()


In [ ]:
# Outlier check before trusting the chart's scale
p99 = delay_review["delivery_delay_days"].quantile(0.99)
p01 = delay_review["delivery_delay_days"].quantile(0.01)
print(f"Delay 1st/99th percentile: {p01:.0f} / {p99:.0f} days")
print(f"Max delay: {delay_review['delivery_delay_days'].max():.0f} days | check this isn't a data-entry error")

# Clip extreme outliers for plotting only (keep raw data for stats below)
delay_review_plot = delay_review[delay_review["delivery_delay_days"].between(p01, p99)]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=delay_review_plot,
    x="review_score",
    y="delivery_delay_days",
    ax=ax,
    color="#DD8452",
)
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_title("Delivery Delay by Review Score (1st-99th percentile)")
ax.set_xlabel("Review Score (1-5)")
ax.set_ylabel("Delay vs. Estimate (days, positive = late)")
plt.tight_layout()
plt.show()


In [ ]:
# Spearman, not Pearson | review score is ordinal
corr, p_value = stats.spearmanr(delay_review["delivery_delay_days"], delay_review["review_score"])
print(f"Spearman correlation: {corr:.3f}  (p = {p_value:.2e})")

on_time = delay_review[delay_review["delivery_delay_days"] <= 0]["review_score"]
very_late = delay_review[delay_review["delivery_delay_days"] > 7]["review_score"]

print(f"On-time/early orders  -> avg review score: {on_time.mean():.2f}  (n={len(on_time)})")
print(f"7+ days late orders   -> avg review score: {very_late.mean():.2f}  (n={len(very_late)})")

t_stat, t_p = stats.ttest_ind(on_time, very_late, equal_var=False)
print(f"t-test p-value: {t_p:.2e}")
print("\nNote: 7-day cutoff is illustrative, not a validated SLA threshold | treat as a sensitivity check, not a fixed rule.")


In [ ]:
# CRITICAL FIX: segment by category to rule out confounding.
# Join the order-grain item aggregate (order_items_agg) | safe, because it's already one row per order.
delay_review_cat = delay_review.merge(
    order_items_agg[["order_id", "primary_category", "freight_total"]], on="order_id", how="left"
)

category_delay_review = (
    delay_review_cat.groupby("primary_category")
    .agg(
        n_orders=("order_id", "count"),
        avg_delay=("delivery_delay_days", "mean"),
        avg_review=("review_score", "mean"),
    )
    .query("n_orders >= 30")  # drop categories too small to trust
    .sort_values("avg_review")
)
category_delay_review.head(15)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    category_delay_review["avg_delay"],
    category_delay_review["avg_review"],
    s=category_delay_review["n_orders"] / 5,
    alpha=0.6,
    c="#C44E52",
)
ax.set_xlabel("Avg. Delivery Delay (days)")
ax.set_ylabel("Avg. Review Score")
ax.set_title("Category-Level: Delay vs. Review Score (bubble size = order count)")
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

cat_corr, cat_p = stats.spearmanr(category_delay_review["avg_delay"], category_delay_review["avg_review"])
print(f"Category-level Spearman correlation: {cat_corr:.3f} (p = {cat_p:.2e})")
print("If this is meaningfully weaker than the order-level correlation above, category is acting as a confounder.")


## 7. Q3 — Regional Delivery Performance

**Fix applied from audit:** filtered to states with a minimum order count so small-n states don't produce misleading, noisy bars.

In [ ]:
MIN_ORDERS_PER_STATE = 50  # states below this are excluded | too few orders for a reliable average

delivered_geo = delay_review.merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left"
)

state_delay_all = (
    delivered_geo.dropna(subset=["delivery_delay_days"])
    .groupby("customer_state")
    .agg(
        avg_delay_days=("delivery_delay_days", "mean"),
        avg_review_score=("review_score", "mean"),
        n_orders=("order_id", "nunique"),
    )
    .sort_values("avg_delay_days", ascending=False)
)

excluded_states = state_delay_all[state_delay_all["n_orders"] < MIN_ORDERS_PER_STATE]
state_delay = state_delay_all[state_delay_all["n_orders"] >= MIN_ORDERS_PER_STATE]

print(f"Excluded {len(excluded_states)} states with fewer than {MIN_ORDERS_PER_STATE} orders: "
      f"{list(excluded_states.index)}")
state_delay


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(
    data=state_delay.reset_index(),
    y="customer_state",
    x="avg_delay_days",
    ax=ax,
    color="#55A868",
)
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_title(f"Average Delivery Delay by State (n \u2265 {MIN_ORDERS_PER_STATE} orders only)")
ax.set_xlabel("Avg. Delay vs. Estimate (days)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 8. Q4 — Seasonality

**Caveat from audit:** Olist's marketplace was growing rapidly through 2016-2018, so raw monthly order counts partly reflect platform growth, not pure seasonal demand. Showing seller count alongside order count to make growth visible rather than implying pure seasonality.

In [ ]:
monthly_orders = (
    orders.dropna(subset=["order_purchase_timestamp"])
    .assign(month=lambda d: d["order_purchase_timestamp"].dt.to_period("M").astype(str))
    .groupby("month")
    .size()
    .reset_index(name="n_orders")
)

monthly_active_sellers = (
    order_items.merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id", how="left")
    .dropna(subset=["order_purchase_timestamp"])
    .assign(month=lambda d: d["order_purchase_timestamp"].dt.to_period("M").astype(str))
    .groupby("month")["seller_id"].nunique()
    .reset_index(name="n_active_sellers")
)

monthly_orders = monthly_orders.merge(monthly_active_sellers, on="month", how="left")
monthly_orders["orders_per_seller"] = monthly_orders["n_orders"] / monthly_orders["n_active_sellers"]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
sns.lineplot(data=monthly_orders, x="month", y="n_orders", marker="o", ax=axes[0])
axes[0].set_title("Monthly Order Volume (raw | includes platform growth)")
axes[0].set_ylabel("Orders")

sns.lineplot(data=monthly_orders, x="month", y="orders_per_seller", marker="o", ax=axes[1], color="#DD8452")
axes[1].set_title("Orders per Active Seller (growth-adjusted proxy)")
axes[1].set_ylabel("Orders / Seller")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Q5 — Payment Type vs. Order Value

In [ ]:
payment_summary = (
    payments.groupby("payment_type")
    .agg(
        n_orders=("order_id", "nunique"),
        avg_value=("payment_value", "mean"),
        avg_installments=("payment_installments", "mean"),
    )
    .sort_values("n_orders", ascending=False)
)
payment_summary


## 10. Freight Cost vs. Review Score

**Added from audit** — `freight_value` was in the raw dataset but wasn't being used. Checking whether shipping cost itself (not just delivery speed) relates to satisfaction.

In [ ]:
delay_review_freight = delay_review.merge(
    order_items_agg[["order_id", "freight_total", "item_price_total"]], on="order_id", how="left"
)
delay_review_freight["freight_pct_of_order"] = (
    delay_review_freight["freight_total"] / (delay_review_freight["item_price_total"] + delay_review_freight["freight_total"])
)

freight_corr, freight_p = stats.spearmanr(
    delay_review_freight["freight_pct_of_order"].dropna(),
    delay_review_freight.loc[delay_review_freight["freight_pct_of_order"].notna(), "review_score"],
)
print(f"Freight-% vs. review score Spearman correlation: {freight_corr:.3f} (p = {freight_p:.2e})")


## 11. Scoped-Out Questions (not fully explored here)

- Seller performance: do high-volume sellers maintain review scores as well as low-volume sellers?
- Repeat purchase rate by category.
- Product weight/dimensions vs. delivery delay.
- Review-score variance by category (inconsistent quality vs. mismatched expectations).


## 12. Key Findings Summary

Fill this in using the actual numbers printed above once you've run the notebook. Each finding should end in a recommendation, not just an observation.

**Finding 1 (category/revenue):** [ ]
**Finding 2 (delay vs. review, order-level AND category-level):** [ ] — state both correlations and note whether category weakens the order-level relationship.
**Finding 3 (regional, n≥50 only):** [ ]
**Finding 4 (freight cost):** [ ]
**Recommendation:** [specific, actionable]

**Limitations:** [dropna %, 7-day cutoff caveat, growth-adjusted seasonality caveat, excluded small-n states]


## 13. Export for Streamlit App

In [ ]:
revenue_by_category.to_csv("revenue_by_category.csv", index=False)
delay_review.to_csv("delay_review.csv", index=False)
category_delay_review.reset_index().to_csv("category_delay_review.csv", index=False)
state_delay.reset_index().to_csv("state_delay.csv", index=False)
monthly_orders.to_csv("monthly_orders.csv", index=False)
payment_summary.reset_index().to_csv("payment_summary.csv", index=False)
delay_review_freight.to_csv("delay_review_freight.csv", index=False)

print("Exported CSVs for the Streamlit app.")
